## Document Forgery Detection: Training + Centralized Evaluation

In [ ]:
import json
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


### 1) Load datasets

In [ ]:
TRAIN_DATASET_PATH = "dataset_outputs/train.csv"
VALIDATION_DATASET_PATH = "dataset_outputs/val.csv"
TEST_DATASET_PATH = "dataset_outputs/test.csv"

if not Path(TRAIN_DATASET_PATH).exists():
    raise FileNotFoundError(f"Missing required dataset: {TRAIN_DATASET_PATH}")
assert Path(VALIDATION_DATASET_PATH).exists(), "Validation dataset is required."
assert Path(TEST_DATASET_PATH).exists(), "Test dataset is required."

df_train_raw = pd.read_csv(TRAIN_DATASET_PATH)
df_validation = pd.read_csv(VALIDATION_DATASET_PATH)
df_test = pd.read_csv(TEST_DATASET_PATH)

print("Training dataset shape:", df_train_raw.shape)
print("Validation dataset shape:", df_validation.shape)
print("Test dataset shape:", df_test.shape)

display(df_train_raw.head())


In [ ]:
LEAKAGE_COLUMNS = ["Document_ID", "Image_Name"]

def drop_leakage_columns(df):
    leakage_prefixes = ("Image_Name_", "Country_Code_", "Country_Name_")
    drop_cols = [
        col for col in df.columns
        if col in LEAKAGE_COLUMNS or any(col.startswith(prefix) for prefix in leakage_prefixes)
    ]
    if drop_cols:
        print("[WARN] Dropping leakage/pre-encoded columns:", sorted(drop_cols))
    return df.drop(columns=drop_cols, errors="ignore")

X_train = drop_leakage_columns(df_train_raw.drop(columns=["Label"]))
y_train = df_train_raw["Label"].astype(int)

X_val = drop_leakage_columns(df_validation.drop(columns=["Label"]))
y_val = df_validation["Label"].astype(int)

X_test = drop_leakage_columns(df_test.drop(columns=["Label"]))
y_test = df_test["Label"].astype(int)

categorical_features = [c for c in ["Country_Code", "Country_Name"] if c in X_train.columns]
numerical_features = [c for c in X_train.columns if c not in categorical_features]

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)


### 2) Train model

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numerical_features,
        ),
    ],
    remainder="drop",
)

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
        ),
    ),
])

model.fit(X_train, y_train)
print("[INFO] Model training complete.")

MODEL_DIR = "trained_models"
os.makedirs(MODEL_DIR, exist_ok=True)
model_path = os.path.join(MODEL_DIR, "forged_document_rf_model.pkl")
joblib.dump(model, model_path)
print(f"[INFO] Trained model saved to: {model_path}")


### 3) Validation evaluation

In [ ]:
# Validation evaluation
y_val_pred = model.predict(X_val)

print("\n===== VALIDATION VS TEST =====")
print(f"Validation Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")


### 📊 MODEL EVALUATION (TEST SET)

In [ ]:
# Predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

specificity = tn / (tn + fp + 1e-6)
fpr = fp / (fp + tn + 1e-6)
roc_auc = roc_auc_score(y_test, y_prob)

print("\n===== MODEL PERFORMANCE =====")
print(f"Accuracy            : {accuracy:.4f}")
print(f"Precision           : {precision:.4f}")
print(f"Recall (Detection)  : {recall:.4f}")
print(f"F1-score            : {f1:.4f}")
print(f"Specificity         : {specificity:.4f}")
print(f"False Positive Rate : {fpr:.4f}")
print(f"ROC-AUC             : {roc_auc:.4f}")

print("\n===== VALIDATION VS TEST =====")
print(f"Validation Accuracy: {accuracy_score(y_val, y_val_pred):.4f}")
print(f"Test Accuracy      : {accuracy:.4f}")

metrics_path = os.path.join(MODEL_DIR, "training_metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({
        "validation_set": {
            "accuracy": float(accuracy_score(y_val, y_val_pred))
        },
        "testing_set": {
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1_score": float(f1),
            "specificity": float(specificity),
            "false_positive_rate": float(fpr),
            "roc_auc": float(roc_auc),
        },
    }, f, indent=2)
print(f"[INFO] Metrics saved to: {metrics_path}")


### 📊 CONFUSION MATRIX VISUALIZATION

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Confusion Matrix")
plt.show()


### 📊 ROC CURVE

In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve")
plt.show()


### 📊 NER + CV SYSTEM EVALUATION

In [ ]:
df_test = pd.read_csv("dataset_outputs/test.csv")

def safe_mean(col):
    return df_test[col].mean() if col in df_test.columns else 0

print("\n===== NER PERFORMANCE =====")
print(f"Field Completeness (Recall): {safe_mean('Field_Completeness'):.4f}")
print(f"NER Precision             : {safe_mean('Precision'):.4f}")
print(f"NER F1-score              : {safe_mean('F1_Score'):.4f}")


### 📊 CV + AML METRICS

In [ ]:
print("\n===== CV + AML METRICS =====")
print(f"OCR Confidence Mean   : {safe_mean('OCR_Quality'):.4f}")
print(f"Risk Score Mean       : {safe_mean('Risk_Score'):.4f}")
print(f"Risk Consistency Mean : {safe_mean('Risk_Consistency'):.4f}")


### 📊 EXTRA INSIGHT

In [ ]:
high_quality = (df_test["Field_Completeness"] > 0.8).mean() if "Field_Completeness" in df_test.columns else 0
print(f"\nHigh-quality extraction rate: {high_quality*100:.2f}%")


### 9) Feature importance

In [ ]:
preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

feature_names = preprocessor.get_feature_names_out()
importances = pd.Series(classifier.feature_importances_, index=feature_names).sort_values(ascending=False)

top_features = importances.nlargest(10)
plt.figure(figsize=(10, 6))
plt.barh(top_features.index[::-1], top_features.values[::-1])
plt.title("Top 10 Forensic Indicators for Forgery Detection")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.show()

top_features
